In [ ]:
!pip install --upgrade google-cloud-bigquery vertexai pandas streamlit pyngrok

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from google.cloud import bigquery
import vertexai
from vertexai.generative_models import GenerativeModel

PROJECT_ID = "ai-workforce-attrition-project"
LOCATION = "europe-north1"

# BigQuery
bq_client = bigquery.Client(project=PROJECT_ID)

# Vertex AI
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION
)

# Gemini
model = GenerativeModel("gemini-2.5-pro")

In [ ]:
# Fetch SQL queries

kpi_query = """select
round(avg(burnout_score),2) as avg_burnout_score,
round(countif(attrition_risk = 'High') * 100.0 / count(*),2) as pct_high_attrition_risk,
round(avg(hours_with_ai_assistance_daily),2) as avg_daily_ai_hours,
round(avg(productivity_score),2) as avg_productivity_score,
round(avg(ai_replaces_my_tasks_pct),2) as tasks_replaced
from my_db.AI_workforce;
"""

kpi_df = bq_client.query(kpi_query).to_dataframe()
kpi_df

In [ ]:
# Task replacement vs role

task_replacement_query = """
select job_role,
round(avg(ai_replaces_my_tasks_pct),2) as tasks_replacement_percent
from my_db.AI_workforce
group by job_role
order by tasks_replacement_percent desc;
"""

task_replacement_df = bq_client.query(task_replacement_query).to_dataframe()
task_replacement_df


In [ ]:
# Attrition risk by AI adoption stage

attrition_risk_query = """
select ai_adoption_stage, attrition_risk,
count(*) as emp_count,
round(count(*) * 100.0/ sum(count(*)) over(partition by ai_adoption_stage),2) as percent
from my_db.AI_workforce
group by ai_adoption_stage, attrition_risk
order by ai_adoption_stage, attrition_risk;
"""

attrition_risk_df = bq_client.query(attrition_risk_query).to_dataframe()
attrition_risk_df


In [ ]:
# Fear of AI replacement

Fear_of_AI_query = """
select fear_of_ai_replacement,
round(count(*) * 100.0 / sum(count(*)) over(),2) as percent
from my_db.AI_workforce
group by fear_of_ai_replacement;
"""

Fear_of_AI_df = bq_client.query(Fear_of_AI_query).to_dataframe()
Fear_of_AI_df



In [ ]:
# High attrition risk cohort

high_attrition_query = """
select industry,
round(avg(years_experience),2) as avg_YoE,
round(avg(burnout_score),2) as avg_burnout,
round(avg(job_satisfaction_1_5),2) as job_satisfaction
from my_db.AI_workforce
where fear_of_ai_replacement = 'High' and attrition_risk = 'High'
group by industry
order by avg_YoE desc;
"""

high_attrition_df = bq_client.query(high_attrition_query).to_dataframe()
high_attrition_df


In [ ]:
# Convert data to text format

kpi_context = kpi_df.to_string(index=False)
high_attrition_context = high_attrition_df.to_string(index=False)
attrition_by_adoption_context = attrition_risk_df.to_string(index=False)
task_replacement_context = task_replacement_df.to_string(index=False)
fear_of_ai_context = Fear_of_AI_df.to_string(index=False)

In [ ]:
# generate_insights function
def generate_insights(user_question):
    prompt = f"""
    You are a senior data analyst.

    Dataset Context:
    - 1500 tech employees (2026)
    - Focus: AI adoption vs burnout, productivity, and attrition

    Key KPIs:
    {kpi_context}

    Industry Attrition Insights:
    {high_attrition_context}

    Attrition Risk vs AI Adoption Stage Insights:
    {attrition_by_adoption_context}

    Percent of tasks replaced by AI for each job role Insights:
    {task_replacement_context}

    Fear of AI replacement Insights:
    {fear_of_ai_context}


    User Question:
    {user_question}
    Give insights, risks, and recommendations based on the user question.

    Instructions:
    1. Answer in 3 - 4 sentences only.
    2. Identify key patterns and trends
    3. Explain business impact
    4. Highlight risks (burnout, attrition, overuse of AI)
    4. Provide clear, actionable recommendations
    5. Keep response structured and concise with headings
    6. Only use the provided data. Do not assume missing values
    """
    response = model.generate_content(prompt)
    return response.text

In [ ]:
user_question = input("Enter your question")

#call the function
response = generate_insights(user_question)


In [ ]:
print (response)

In [ ]:
## Streamlit app

%%writefile app.py
import streamlit as st
import base64
from google.cloud import bigquery
import vertexai
from vertexai.generative_models import GenerativeModel

# -----------------------------
# INIT
# -----------------------------
# Set your project details
PROJECT_ID = "neat-dynamo-507714-e4"
LOCATION = "us-central1"

# Initialize clients
bq_client = bigquery.Client(project=PROJECT_ID)
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Load model
model = GenerativeModel("gemini-2.5-pro")

#-------- Load Data Context -------------

# fetch KPIs query
kpi_query = """
select
  round(avg(burnout_score),1) as avg_burnout_score,
  round(countif(attrition_risk = 'High') * 100.0 / count(*), 1) as pct_high_attrition_risk,
  round(avg(hours_with_ai_assistance_daily),1) as avg_daily_ai_hours,
  round(avg(productivity_score),1) as avg_productivity_score,
  round(avg(ai_replaces_my_tasks_pct),1) as tasks_replaced
from my_db.workforce;
"""

kpi_df = bq_client.query(kpi_query).to_dataframe()

#high attrition risk cohort
high_attrition_query = """
--High Attrition Risk Cohort
select industry, round(avg(years_experience),2) as avg_YoE,
round(avg(burnout_score),2) as avg_burnout,
round(avg(job_satisfaction_1_5),2) as avg_job_satisfaction
from my_db.workforce
where fear_of_ai_replacement = 'High' and attrition_risk = 'High'
group by industry
order by avg_YoE desc;

"""

high_attrition_df = bq_client.query(high_attrition_query).to_dataframe()


#attrition risk by ai adoption
attrition_by_adoption_query= """
--Attrition Risk By AI Adoption Stage
select ai_adoption_stage, attrition_risk,
count(*) as emp_count,
round(count(*) * 100.0/ sum(count(*)) over(partition by ai_adoption_stage),2) as percent
from my_db.workforce
group by ai_adoption_stage, attrition_risk
order by ai_adoption_stage, attrition_risk;

"""

attrition_by_adoption_df = bq_client.query(attrition_by_adoption_query).to_dataframe()

#tasks replaced by AI vs role
task_replacement_query = """
--task replacement % by role
select job_role, round(avg(ai_replaces_my_tasks_pct),1) as task_replacement_percent
from my_db.workforce
group by job_role
order by  task_replacement_percent desc;
"""

task_replacement_df = bq_client.query(task_replacement_query).to_dataframe()

#fear of ai replacement
fear_of_ai_query = """
select fear_of_ai_replacement, round(count(*) * 100.0/ sum(count(*)) over (), 2) as percent
from my_db.workforce
group by fear_of_ai_replacement;
"""

fear_of_ai_df = bq_client.query(fear_of_ai_query).to_dataframe()


#Convert Data to Text Context
kpi_context = kpi_df.to_string(index=False)
high_attrition_context = high_attrition_df.to_string(index=False)
attrition_by_adoption_context = attrition_by_adoption_df.to_string(index=False)
task_replacement_context =  task_replacement_df.to_string(index=False)
fear_of_ai_context = fear_of_ai_df.to_string(index=False)


# generate_insights function
def generate_insights(user_question):
    prompt = f"""
    You are a senior data analyst.

    Dataset Context:
    - 1500 employees (2026)
    - Focus: AI adoption vs burnout, productivity, and attrition

    Key KPIs:
    {kpi_context}

    Industry Attrition Insights:
    {high_attrition_context}

    Attrition Risk vs AI Adoption Stage Insights:
    {attrition_by_adoption_context}

    Percent of tasks replaced by AI for each job role Insights:
    {task_replacement_context}

    Fear of AI replacement Insights:
    {fear_of_ai_context}


    User Question:
    {user_question}

    Instructions to answer the user question:
    1. Answer straightforward in 3 - 4 sentences only
    2. Explain business impact
    3. Provide clear, actionable recommendations
    4. Keep response structured and concise with headings
    5. Only use the provided data. Do not assume missing values
    """
    response = model.generate_content(prompt)
    return response.text

#--- UI -----
st.set_page_config(layout="wide")

st.title("AI Workforce Dashboard + GenAI Insights")

# ---- LOAD DASHBOARD ----
with open("AI Workplace Attrition.pdf", "rb") as f:
    pdf_bytes = f.read()

base64_pdf = base64.b64encode(pdf_bytes).decode('utf-8')

pdf_display = f"""
<iframe src="data:application/pdf;base64,{base64_pdf}"
width="100%" height="600" type="application/pdf"></iframe>
"""

st.markdown("## Dashboard")
st.markdown(pdf_display, unsafe_allow_html=True)


# ---- QUESTION INPUT ----
st.markdown("## 🤖 Ask a question about the dashboard")

user_question = st.text_input("Enter your question:")

if st.button("Get Insights"):
    if user_question:
        st.write("### Response:")
        #from ai_workforce_attrition import generate_insights

        response = generate_insights(user_question)
        st.write(response)

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3JpqCUXKzDl7Twe75cDhsXTXoiM_2kZHgP6pzCu8Szzm1doAY")

In [ ]:
!pkill -f streamlit
!streamlit run app.py &>/dev/null &

In [ ]:
public_url = ngrok.connect(8501)
print(public_url)